<a href="https://colab.research.google.com/github/Prathamesh29k/Devflow-ai/blob/main/RAG_Students_Example_1_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 4 — Practical RAG: Enterprise HR Knowledge Assistant

## Student Notebook — FREE Google Colab

We will build the syllabus project step by step:

**PDFs → Text → Chunks → Metadata → Embeddings → Vector Database → Similarity Search → Semantic Search → RAG → Answer → Citations → Multi-document Search → Evaluation**

No OpenAI API credits are required.

## How to use this notebook

For each stage:

**1. Understand the concept → 2. Run the code → 3. Observe the result → 4. Move to the next stage.**

The application is the practical demonstration of every syllabus topic.

# 1. Install the required libraries

- `pypdf` → PDF text extraction
- `sentence-transformers` → free local embeddings
- `chromadb` → local vector database
- `transformers` → local LLM
- `accelerate` → model runtime

No paid API is needed.

In [1]:
!pip install -q pypdf sentence-transformers chromadb transformers accelerate
print("✅ Installation complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.

# 2. Upload HR Policy PDFs

Upload one or more PDFs, for example:

- Leave and Holiday Policy
- Work From Home Policy
- Employee Handbook
- Travel Policy

Multiple PDFs let us demonstrate **multi-document search**.

In [2]:
from google.colab import files

uploaded_files = files.upload()

print(f"✅ Uploaded {len(uploaded_files)} file(s)")
for filename in uploaded_files:
    print(" -", filename)

Saving Restaurant_Food_Quality_Report.pdf to Restaurant_Food_Quality_Report.pdf
✅ Uploaded 1 file(s)
 - Restaurant_Food_Quality_Report.pdf


# 3. PDF → Text

First we extract text from every PDF page.

We also keep **metadata**:

- `source` = PDF filename
- `page` = page number

This metadata will later be used for citations.

In [3]:
from pypdf import PdfReader

documents = []

for filename in uploaded_files:
    reader = PdfReader(filename)

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        if text and text.strip():
            documents.append({
                "text": text.strip(),
                "source": filename,
                "page": page_number
            })

print("✅ Text extraction complete")
print("Pages extracted:", len(documents))

for doc in documents[:2]:
    print("\nSOURCE:", doc["source"])
    print("PAGE:", doc["page"])
    print("TEXT PREVIEW:", doc["text"][:500])

✅ Text extraction complete
Pages extracted: 7

SOURCE: Restaurant_Food_Quality_Report.pdf
PAGE: 1
TEXT PREVIEW: Restaurant Food Quality Report    Page 1
 RESTAURANT FOOD QUALITY
 How Good Is the Food?
 A practical framework for evaluating taste, freshness, presentation, hygiene, service, value, and customer
 satisfaction.
Evaluation Area
What We Look For
Taste
Flavor balance, seasoning, aroma, consistency
Freshness
Ingredient quality, texture, preparation
Presentation
Plating, appearance, portion appeal
Hygiene
Clean kitchen, utensils, storage and serving
Service
Speed, accuracy, staff helpfulness
Value


SOURCE: Restaurant_Food_Quality_Report.pdf
PAGE: 2
TEXT PREVIEW: Restaurant Food Quality Report    Page 2
1. What Makes Restaurant Food Good?
Good restaurant food is more than simply tasting delicious. A strong dining experience combines flavor,
freshness, texture, aroma, temperature, appearance, and consistency.
Taste and Flavor
A well-prepared dish should have balanced seasoning. 

# 4. Chunking

### Why chunk?

A large PDF may contain thousands of words. We divide it into smaller searchable pieces called **chunks**.

For this classroom lab:

- Chunk size = 500 words
- Overlap = 50 words

Each chunk keeps its `source` and `page` metadata.

In [4]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

chunks = []
chunk_id = 0

for document in documents:
    words = document["text"].split()
    start = 0

    while start < len(words):
        end = start + CHUNK_SIZE
        chunk_text = " ".join(words[start:end])

        if chunk_text.strip():
            chunks.append({
                "chunk_id": chunk_id,
                "text": chunk_text,
                "source": document["source"],
                "page": document["page"]
            })
            chunk_id += 1

        start += CHUNK_SIZE - CHUNK_OVERLAP

print("✅ Chunking complete")
print("Total chunks:", len(chunks))

✅ Chunking complete
Total chunks: 7


In [ ]:
for chunk in chunks[:3]:
    print("=" * 60)
    print("CHUNK:", chunk["chunk_id"])
    print("SOURCE:", chunk["source"])
    print("PAGE:", chunk["page"])
    print("TEXT:", chunk["text"][:400])

CHUNK: 0
SOURCE: Leave-and-Holiday-Policy.pdf
PAGE: 1
TEXT: KB0044163 - Latest version HR India – Leave and Holiday Policy (PIL) Purpose The purpose of this Policy is to facilitate effective administration and management of employees' leave. To highlight the procedures, benefits, and responsibilities and specify the Company norms for availing leave. Scope This outlines the eligibility principles, specific changes to work methods, benefits, and general guid
CHUNK: 1
SOURCE: Leave-and-Holiday-Policy.pdf
PAGE: 2
TEXT: Leave Description Rules Approvals Privilege Leave Employees are allowed to take leave to attend to personal commitments or go on vacation and spend time with their family in order to maintain work life balance. It could be short term or long-term leave, ranging from a day, a week or more. Such leave is normally planned in advance. Entitlement: 22 Days per annum This leave is proportionally earned 
CHUNK: 2
SOURCE: Leave-and-Holiday-Policy.pdf
PAGE: 3
TEXT: Paternity leave As

# 5. Embeddings

An **embedding** converts text into a numerical vector representing its meaning.

We use the free local model:

`all-MiniLM-L6-v2`

So every chunk can be compared with an employee question without an API call.

In [5]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("✅ Embeddings created")
print("Number of vectors:", len(embeddings))
print("Vector dimensions:", len(embeddings[0]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings created
Number of vectors: 7
Vector dimensions: 384


# 6. Vector Database

We need to store:

- chunks
- embeddings
- metadata

We use **ChromaDB**, a local vector database.

Production systems can use PostgreSQL + pgvector or another vector database.

In [6]:
import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="hr_knowledge"
)

collection.add(
    ids=[str(chunk["chunk_id"]) for chunk in chunks],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[
        {
            "source": chunk["source"],
            "page": chunk["page"]
        }
        for chunk in chunks
    ]
)

print("✅ Vector database ready")
print("Stored chunks:", collection.count())

✅ Vector database ready
Stored chunks: 7


# 7. Ask a Natural-Language Question

Now we move to the query side.

Example:

> How many days of annual leave can I take?

The question will also be converted into an embedding.

In [8]:
question = input("Ask an HR question: ")

query_embedding = embedding_model.encode(
    [question]
)[0].tolist()

print("\nQUESTION:", question)
print("✅ Query embedding created")

Ask an HR question: what makes restuarant food good?

QUESTION: what makes restuarant food good?
✅ Query embedding created


# 8. Similarity Search + Semantic Search

**Similarity Search:** find vectors closest to the question vector.

**Semantic Search:** because embeddings represent meaning, relevant text can be found even when the exact words differ.

We retrieve the Top-K most relevant chunks.

In [9]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=min(3, len(chunks))
)

print("✅ Similarity search complete")

✅ Similarity search complete


In [14]:
retrieved_chunks = []

for i, text in enumerate(results["documents"][0]):
    metadata = results["metadatas"][0][i]
    distance = results["distances"][0][i]

    item = {
        "text": text,
        "source": metadata["source"],
        "page": metadata["page"],
        "distance": distance
    }

    retrieved_chunks.append(item)

    print("=" * 60)
    print("RANK:", i + 1)
    print("DISTANCE:", round(distance, 4))
    print("SOURCE:", metadata["source"])
    print("PAGE:", metadata["page"])
    print("TEXT:", text[:700])

RANK: 1
DISTANCE: 0.9322
SOURCE: Restaurant_Food_Quality_Report.pdf
PAGE: 2
TEXT: Restaurant Food Quality Report  Page 2 1. What Makes Restaurant Food Good? Good restaurant food is more than simply tasting delicious. A strong dining experience combines flavor, freshness, texture, aroma, temperature, appearance, and consistency. Taste and Flavor A well-prepared dish should have balanced seasoning. Salt, sweetness, acidity, spice, bitterness, and umami should support each other rather than overpower the main ingredients. Fresh Ingredients Fresh vegetables should retain appropriate color and texture, proteins should be handled correctly, and sauces or accompaniments should taste clean rather than stale. Texture and Temperature Texture creates contrast and enjoyment: crisp w
RANK: 2
DISTANCE: 1.0897
SOURCE: Restaurant_Food_Quality_Report.pdf
PAGE: 7
TEXT: Restaurant Food Quality Report  Page 7 6. Final Assessment & Recommendations A restaurant can be considered genuinely good when it con

# 9. Retrieval

This is the **Retrieval** part of RAG.

```text
Employee Question
       ↓
Query Embedding
       ↓
Similarity Search
       ↓
Top-K Relevant Chunks
```

The system has found evidence before generating an answer.

# 10. Augmentation

Now we put the retrieved evidence into a prompt.

This is the **Augmentation** part of RAG.

The language model will receive:

1. The employee's question
2. The retrieved company policy information

In [15]:
context = "\n\n".join(
    f"SOURCE: {chunk['source']}\n"
    f"PAGE: {chunk['page']}\n"
    f"CONTENT: {chunk['text']}"
    for chunk in retrieved_chunks
)

prompt = (
    "You are an HR Policy Assistant.\n\n"
    "Answer the question using ONLY the company policy context. "
    "Do not invent policies.\n\n"
    "QUESTION:\n" + question + "\n\n"
    "COMPANY POLICY CONTEXT:\n" + context
)

print("✅ RAG prompt created")
print(prompt[:3500])

✅ RAG prompt created
You are an HR Policy Assistant.

Answer the question using ONLY the company policy context. Do not invent policies.

QUESTION:
what makes restuarant food good?

COMPANY POLICY CONTEXT:
SOURCE: Restaurant_Food_Quality_Report.pdf
PAGE: 2
CONTENT: Restaurant Food Quality Report  Page 2 1. What Makes Restaurant Food Good? Good restaurant food is more than simply tasting delicious. A strong dining experience combines flavor, freshness, texture, aroma, temperature, appearance, and consistency. Taste and Flavor A well-prepared dish should have balanced seasoning. Salt, sweetness, acidity, spice, bitterness, and umami should support each other rather than overpower the main ingredients. Fresh Ingredients Fresh vegetables should retain appropriate color and texture, proteins should be handled correctly, and sauces or accompaniments should taste clean rather than stale. Texture and Temperature Texture creates contrast and enjoyment: crisp with soft, creamy with crunchy, or 

# 11. Generation — Local LLM

Now we perform **Generation**.

We use a small instruction-following model locally in Colab.

If available, use a GPU:

**Runtime → Change runtime type → T4 GPU**

This keeps the practical free of API charges.

In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading local LLM on:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = model.to(device)
model.eval()

print("✅ Local LLM loaded")

Loading local LLM on: cpu


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Local LLM loaded


In [17]:
messages = [
    {
        "role": "system",
        "content": "You are an HR policy assistant. Use only the supplied company context. Do not invent policies."
    },
    {
        "role": "user",
        "content": prompt
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=4096
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False
    )

new_tokens = output[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("ANSWER")
print("=" * 60)
print(answer)

ANSWER
Restaurateur's role in ensuring their restaurant food meets high standards involves several key elements:

1. **Taste and Flavor**: A well-prepared dish should have balanced seasoning, supporting flavors such as salt, sweetness, acidity, spice, bitterness, and umami. Fresh ingredients should retain their color and texture, proteins handled correctly, and sauces or accompaniments tasting clean without being stale.

2. **Texture and Temperature**: Texture creates contrast and enjoyment; crisp with soft, creamy with crunchy, or tender with a light crust. Food should also arrive at a suitable serving temperature.

3. **Consistency**: A restaurant earns trust when the same signature dishes maintain similar quality across different visits. Consistency is crucial for maintaining professionalism in kitchen operations.

4. **Key Idea**: Great food should satisfy the senses while remaining safe, fresh, and reliably prepared.

**Final Assessment & Recommendations**:
- **Best Dish**: Mainta

# 12. Citations

Because we kept `source` and `page` as metadata, we can show where the retrieved evidence came from.

This gives the user a **source-backed answer**.

In [18]:
print("SOURCES USED")
print("-" * 50)

seen = set()

for chunk in retrieved_chunks:
    citation = (chunk["source"], chunk["page"])

    if citation not in seen:
        print(f"• {chunk['source']} — Page {chunk['page']}")
        seen.add(citation)

SOURCES USED
--------------------------------------------------
• Restaurant_Food_Quality_Report.pdf — Page 2
• Restaurant_Food_Quality_Report.pdf — Page 7
• Restaurant_Food_Quality_Report.pdf — Page 1


# 13. Multi-Document Search

All uploaded PDFs are stored in the same vector database.

Therefore one employee question can search across:

```text
Leave Policy
Work From Home Policy
Employee Handbook
Travel Policy
        ↓
   One Vector DB
        ↓
Relevant chunks
```

Try questions about different policies and observe the source documents returned.

In [19]:
print("Documents in the knowledge base:")

for source in sorted(set(chunk["source"] for chunk in chunks)):
    print(" -", source)

print("\nTotal searchable chunks:", collection.count())

Documents in the knowledge base:
 - Restaurant_Food_Quality_Report.pdf

Total searchable chunks: 7


# 14. Evaluation

A RAG system must be evaluated.

### Retrieval
Did we retrieve the correct chunks?

### Answer
Did the answer correctly use those chunks?

### Groundedness
Are the answer's claims supported by the retrieved documents?

### Citation correctness
Does the cited document/page actually support the answer?

In [20]:
print("Retrieved evidence for evaluation:")
for rank, chunk in enumerate(retrieved_chunks, start=1):
    print(
        f"{rank}. {chunk['source']} | "
        f"Page {chunk['page']} | "
        f"distance={chunk['distance']:.4f}"
    )

Retrieved evidence for evaluation:
1. Restaurant_Food_Quality_Report.pdf | Page 2 | distance=0.9322
2. Restaurant_Food_Quality_Report.pdf | Page 7 | distance=1.0897
3. Restaurant_Food_Quality_Report.pdf | Page 1 | distance=1.1056


# 15. Final RAG Pipeline

```text
PDFs
 ↓
Text Extraction
 ↓
Chunking
 ↓
Metadata
 ↓
Embeddings
 ↓
Vector Database
 ↓
User Question
 ↓
Query Embedding
 ↓
Similarity / Semantic Search
 ↓
Relevant Chunks
 ↓
Augmented Prompt
 ↓
LLM Generation
 ↓
Source-backed Answer
 ↓
Citations
```

## Syllabus coverage

- ✅ Embeddings
- ✅ Vector Databases
- ✅ Semantic Search
- ✅ Chunking Strategies
- ✅ Metadata
- ✅ Similarity Search
- ✅ RAG Pipelines
- ✅ Evaluation

## Hands-on features

- ✅ Upload PDFs
- ✅ Ask natural-language questions
- ✅ Source-backed answers
- ✅ View citations
- ✅ Multi-document search

## Industry Challenge

**HR policy assistant capable of answering employee queries using company documents.**